In [30]:
"""
00_data_collection.py
=====================
BIM Sign Language — Data Collection via Desktop Camera

CAMERA SETUP:
  Uses the default desktop/laptop camera (Index 0).
  Rotation is disabled by default since desktop cameras are horizontal.

RESOLUTION SITUATION:
  Phone cameras are Portrait, Desktop cameras are Landscape. 
  To match the Phone's 1080x1248 processing frame (Aspect Ratio ~0.865), 
  this script takes a CENTER CROP of the desktop webcam.
  
  This ensures the keypoints (which MediaPipe normalizes to [0,1])
  are physically identical whether collected on a phone or desktop. ✅
"""

import cv2
import mediapipe as mp
import numpy as np
import os
import time

# ==============================
# DESKTOP CAMERA CONFIG
# ==============================
WEBCAM_INDEX = 0  

# Disabled for desktop webcams
ROTATE_FRAME  = False
ROTATION_CODE = cv2.ROTATE_90_COUNTERCLOCKWISE 

# ==============================
# CROP CONFIG
# ==============================
CAMERA_FRACTION = 0.65

# The target aspect ratio matches the Phone setup: 1080 width / (1920 * 0.65 height)
TARGET_ASPECT_RATIO = 1080 / (1920 * CAMERA_FRACTION)

# Flutter sensor dimensions (for display resize reference)
FLUTTER_SENSOR_W = 480
FLUTTER_SENSOR_H = 720
FLUTTER_CROP_H   = int(FLUTTER_SENSOR_H * CAMERA_FRACTION)  # 468

# ==============================
# CONFIG — keep in sync with 02_train_model.py
# ==============================
DATA_PATH       = "dataset_desktop"
SEQUENCE_LENGTH = 8

BASE_TARGET_SEQUENCES = 10
COUNTDOWN_SECONDS     = 3

HAND_LANDMARKS = 21
HAND_VEC_LEN   = HAND_LANDMARKS * 3
HAND_FEATURES  = HAND_VEC_LEN * 2   # 126

SELECTED_FACE_IDX = [
    13, 14,
    78, 308, 82, 312,
    33, 133,
    362, 263,
    70, 63, 105, 66, 107,
    336, 296, 334, 293, 300
]
FACE_FEATURES = len(SELECTED_FACE_IDX) * 3   # 60

POSE_IDX      = [11, 13, 15, 12, 14, 16]
POSE_FEATURES = len(POSE_IDX) * 3             # 18

FEATURE_SIZE = HAND_FEATURES + FACE_FEATURES + POSE_FEATURES  # 204

# Changed from a hard threshold to dynamic based on active regions
MIN_NONZERO_RATIO = 0.40

DEBUG_FOLDER  = "debug_frames"
DEBUG_EVERY_N = 10

SIGN_DURATION_MS = 2000   # default — adjust per sign
SAMPLE_INTERVAL_MS = SIGN_DURATION_MS / SEQUENCE_LENGTH   # e.g. 250ms per frame

_sample_times = []
state_start_time = None
_last_sample_time = 0.0

# ==============================
# SETUP
# ==============================
os.makedirs(DATA_PATH,    exist_ok=True)
os.makedirs(DEBUG_FOLDER, exist_ok=True)

# ==============================
# MIRROR AUGMENTATION
# ==============================
def mirror_sequence(sequence):
    seq_m = sequence.copy()
    for t in range(seq_m.shape[0]):
        frame = seq_m[t]
        for i in range(0, FEATURE_SIZE, 3):
            frame[i] = 1.0 - frame[i]
        left_hand  = frame[0:HAND_VEC_LEN].copy()
        right_hand = frame[HAND_VEC_LEN:HAND_FEATURES].copy()
        frame[0:HAND_VEC_LEN]             = right_hand
        frame[HAND_VEC_LEN:HAND_FEATURES] = left_hand
        pose_start = HAND_FEATURES + FACE_FEATURES
        pose_block = frame[pose_start:pose_start + POSE_FEATURES].copy()
        left_arm   = pose_block[0:9].copy()
        right_arm  = pose_block[9:18].copy()
        pose_block[0:9]  = right_arm
        pose_block[9:18] = left_arm
        frame[pose_start:pose_start + POSE_FEATURES] = pose_block
    return seq_m

# ==============================
# DEBUG FRAME SAVER
# ==============================
mp_drawing        = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_holistic_mod   = mp.solutions.holistic

def save_debug_frame(raw_frame, results, label, counter, use_face, use_pose, crop_info=None):
    raw_path = os.path.join(DEBUG_FOLDER, f"raw_{label}_{counter}.jpg")
    cv2.imwrite(raw_path, raw_frame)

    overlay = raw_frame.copy()
    h, w    = overlay.shape[:2]

    if results.left_hand_landmarks:
        mp_drawing.draw_landmarks(
            overlay, results.left_hand_landmarks,
            mp_holistic_mod.HAND_CONNECTIONS,
            mp_drawing_styles.get_default_hand_landmarks_style())
    if results.right_hand_landmarks:
        mp_drawing.draw_landmarks(
            overlay, results.right_hand_landmarks,
            mp_holistic_mod.HAND_CONNECTIONS,
            mp_drawing_styles.get_default_hand_landmarks_style())

    if use_pose and results.pose_landmarks:
        for idx in POSE_IDX:
            lm    = results.pose_landmarks.landmark[idx]
            cx_px = int(lm.x * w)
            cy_px = int(lm.y * h)
            cv2.circle(overlay, (cx_px, cy_px), 6, (255, 100, 0), -1)
            cv2.putText(overlay, str(idx), (cx_px + 5, cy_px - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 200, 0), 1)

    if use_face and results.face_landmarks:
        for idx in SELECTED_FACE_IDX:
            lm    = results.face_landmarks.landmark[idx]
            cx_px = int(lm.x * w)
            cy_px = int(lm.y * h)
            cv2.circle(overlay, (cx_px, cy_px), 3, (0, 255, 0), -1)

    detected = []
    if results.left_hand_landmarks:  detected.append("L.Hand")
    if results.right_hand_landmarks: detected.append("R.Hand")
    if use_face and results.face_landmarks: detected.append("Face")
    if use_pose and results.pose_landmarks: detected.append("Pose")

    cv2.putText(overlay,
                f"#{counter} | {label} | {w}x{h}px",
                (8, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
    cv2.putText(overlay,
                f"Detected: {', '.join(detected) if detected else 'NONE'}",
                (8, 44), cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                (0, 255, 0) if len(detected) >= 1 else (0, 0, 255), 2)
    if crop_info:
        cv2.putText(overlay, f"Crop: {crop_info}",
                    (8, 66), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1)

    overlay_path = os.path.join(DEBUG_FOLDER, f"overlay_{label}_{counter}.jpg")
    cv2.imwrite(overlay_path, overlay)

# ==============================
# CAMERA INIT
# ==============================
print("=" * 60)
print("  BIM Sign Language — Desktop Data Collection")
print("=" * 60)
print(f"\n💻 Connecting to WebCam Index: {WEBCAM_INDEX}")

cap = cv2.VideoCapture(WEBCAM_INDEX)
if not cap.isOpened():
    print("❌ Cannot connect to Desktop Camera.")
    print("   Make sure no other application is using your webcam.")
    exit(1)
print("✅ Connected\n")

ret, test_frame = cap.read()
if not ret:
    print("❌ Cannot read frame — check connection")
    exit(1)

if ROTATE_FRAME:
    test_frame = cv2.rotate(test_frame, ROTATION_CODE)

raw_h, raw_w = test_frame.shape[:2]
crop_h  = raw_h
crop_w  = int(raw_h * TARGET_ASPECT_RATIO)
start_x = (raw_w - crop_w) // 2
end_x   = start_x + crop_w
crop_info = (f"{crop_w}×{crop_h}px (center crop from {raw_w}×{raw_h})")

DISPLAY_W = FLUTTER_SENSOR_W
DISPLAY_H = FLUTTER_CROP_H

# ==============================
# MODE & LABEL SETUP
# ==============================
print("Select mode:")
print("  1 — Static sign  (hold the pose)")
print("  2 — Dynamic sign (moving gesture)")
mode       = input("Enter 1 or 2: ").strip()
is_dynamic = (mode == "2")

FRAME_SKIP = 4 if is_dynamic else 1

existing_categories = sorted([
    d for d in os.listdir(DATA_PATH)
    if os.path.isdir(os.path.join(DATA_PATH, d))
])

if existing_categories:
    print("\nExisting categories:")
    for i, cat in enumerate(existing_categories, start=1):
        print(f"  {i} — {cat}")
    print("  0 — Create new category")
    choice = input("Select category number, or 0 for new: ").strip()
    if choice.isdigit() and 1 <= int(choice) <= len(existing_categories):
        category = existing_categories[int(choice) - 1]
    else:
        category = input("Enter new category name: ").strip()
else:
    category = input("No categories yet. Enter new category name: ").strip()

category_path = os.path.join(DATA_PATH, category)
os.makedirs(category_path, exist_ok=True)

label      = input("Enter sign label: ").strip()
label_path = os.path.join(category_path, label)
os.makedirs(label_path, exist_ok=True)

# ── NEW: REGION TOGGLES ───────────────────────────────────────────────────
print("\nConfigure required regions for this sign:")
use_face_in = input("  Track Face? (y/n) [Default: y]: ").strip().lower()
use_face = False if use_face_in == 'n' else True

use_pose_in = input("  Track Pose/Arms? (y/n) [Default: y]: ").strip().lower()
use_pose = False if use_pose_in == 'n' else True

# ─────────────────────────────────────────────────────────
# 🛑 NEW: SAVE METADATA
# ─────────────────────────────────────────────────────────
import json
meta_path = os.path.join(label_path, "meta.json")
meta_data = {
    "is_dynamic": is_dynamic,
    "tracked_face": use_face,
    "tracked_pose": use_pose
}
with open(meta_path, "w") as f:
    json.dump(meta_data, f, indent=2)
# ─────────────────────────────────────────────────────────


# Calculate expected feature size to adjust the quality threshold dynamically
active_features = HAND_FEATURES + (FACE_FEATURES if use_face else 0) + (POSE_FEATURES if use_pose else 0)

existing_files = [f for f in os.listdir(label_path) if f.endswith(".npy")]
if existing_files:
    counters = [
        int(f.split("_")[-1].split(".")[0])
        for f in existing_files
        if f.split("_")[-1].split(".")[0].isdigit()
    ]
    counter = (max(counters) + 1) if counters else 0
else:
    counter = 0

target_sequences = counter + BASE_TARGET_SEQUENCES if not is_dynamic else None

print(f"\n✅ Setup complete")
print(f"   Mode             : {'Dynamic' if is_dynamic else 'Static'}")
print(f"   Category         : {category}")
print(f"   Label            : {label}")
print(f"   Tracking Face    : {'✅ Yes' if use_face else '❌ No'}")
print(f"   Tracking Pose    : {'✅ Yes' if use_pose else '❌ No'}")
print("\nPress SPACE to begin. Press ESC to quit.")

# ==============================
# CAPTURE LOOP
# ==============================
sequence          = []
prev_time         = 0
started           = False
waiting_for_space = False
frame_counter     = 0
state             = "IDLE"
countdown_start_time = None

_last_debug_frame   = None
_last_debug_results = None

with mp_holistic_mod.Holistic(
    static_image_mode=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as holistic:

    while True:
        ret, frame = cap.read()
        if not ret: break

        frame_counter += 1

        if ROTATE_FRAME:
            frame = cv2.rotate(frame, ROTATION_CODE)
        
        h_orig, w_orig = frame.shape[:2]

        portrait_w = int(h_orig * 0.75)
        start_x = (w_orig - portrait_w) // 2
        frame = frame[0:h_orig, start_x : start_x + portrait_w]

        h_portrait, w_portrait = frame.shape[:2]
        crop_h = int(h_portrait * CAMERA_FRACTION)
        frame = frame[0:crop_h, 0:w_portrait]

        frame = cv2.flip(frame, 1)

        keypoints = np.zeros(FEATURE_SIZE, dtype=np.float32)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(rgb)
            
        for h_idx, hand in enumerate([results.left_hand_landmarks,
                                       results.right_hand_landmarks]):
            if hand:
                base = h_idx * HAND_VEC_LEN
                for i, lm in enumerate(hand.landmark):
                    keypoints[base + i * 3]     = lm.x
                    keypoints[base + i * 3 + 1] = lm.y
                    keypoints[base + i * 3 + 2] = lm.z
                mp_drawing.draw_landmarks(frame, hand, mp_holistic_mod.HAND_CONNECTIONS)

        # ── NEW: CONDITIONAL EXTRACTION ─────────────────────────────────────
        if use_face and results.face_landmarks:
            h_px, w_px, _ = frame.shape
            for i, idx in enumerate(SELECTED_FACE_IDX):
                lm     = results.face_landmarks.landmark[idx]
                offset = HAND_FEATURES + i * 3
                keypoints[offset]     = lm.x
                keypoints[offset + 1] = lm.y
                keypoints[offset + 2] = lm.z
                cv2.circle(frame, (int(lm.x * w_px), int(lm.y * h_px)), 2, (0, 255, 0), -1)

        if use_pose and results.pose_landmarks:
            h_px, w_px, _ = frame.shape
            pose_start = HAND_FEATURES + FACE_FEATURES
            for i, idx in enumerate(POSE_IDX):
                lm     = results.pose_landmarks.landmark[idx]
                offset = pose_start + i * 3
                keypoints[offset]     = lm.x
                keypoints[offset + 1] = lm.y
                keypoints[offset + 2] = lm.z
                cv2.circle(frame, (int(lm.x * w_px), int(lm.y * h_px)), 5, (255, 100, 0), -1)

        # Calculate quality ratio based only on the ACTIVE features, not the whole 204 array
        non_zero_ratio = np.count_nonzero(keypoints) / active_features

        _last_debug_frame   = frame.copy()
        _last_debug_results = results

        display = cv2.resize(frame, (DISPLAY_W, DISPLAY_H))

        curr_time = time.time()
        fps = 1 / (curr_time - prev_time) if prev_time != 0 else 0
        prev_time = curr_time
        
        def draw_info(extra_y=0):
            cv2.putText(display,
                        f"{'Dynamic' if is_dynamic else 'Static'} | "
                        f"{category}/{label}",
                        (8, 22), cv2.FONT_HERSHEY_SIMPLEX,
                        0.50, (255, 255, 255), 1)
            cv2.putText(display, f"FPS:{int(fps)}",
                        (8, 40 + extra_y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.40, (255, 200, 0), 1)
            cv2.putText(display, f"NZ:{non_zero_ratio:.0%}",
                        (72, 40 + extra_y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.40,
                        (0, 255, 0) if non_zero_ratio >= MIN_NONZERO_RATIO
                        else (0, 0, 255), 1)
            cv2.putText(display,
                        f"Proc:{crop_w}x{crop_h} "
                        f"| View:{DISPLAY_W}x{DISPLAY_H}",
                        (8, 54 + extra_y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.32, (180, 180, 255), 1)
            cv2.putText(display, "ESC=exit",
                        (8, 66 + extra_y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.32, (0, 0, 255), 1)

        def save_sequence(seq_array):
            global counter
            path = os.path.join(label_path, f"{label}_{counter}.npy")
            np.save(path, seq_array)
            # Use active_features for debug logging accuracy
            nz = np.count_nonzero(seq_array) / (active_features * SEQUENCE_LENGTH)
            print(f"  ✅ Saved #{counter}  shape={seq_array.shape}  nz={nz:.0%}")
            if (counter + 1) % DEBUG_EVERY_N == 0:
                if _last_debug_frame is not None:
                    save_debug_frame(
                        _last_debug_frame, _last_debug_results,
                        label, counter, use_face, use_pose, crop_info=crop_info)
            counter += 1

        # ══════════════════════════════════════════════════════════════════
        # STATIC MODE
        # ══════════════════════════════════════════════════════════════════
        if not is_dynamic:
            if not started:
                cv2.putText(display, "Press SPACE to start",
                            (8, 100), cv2.FONT_HERSHEY_SIMPLEX,
                            0.6, (0, 255, 255), 1)
                draw_info()
                cv2.imshow("BIM Data Collection — Desktop view", display)
                key = cv2.waitKey(1)
                if key == 32: started = True
                elif key == 27: break
                continue

            if waiting_for_space:
                cv2.putText(display,
                            f"Done {counter}! SPACE=more  ESC=exit",
                            (8, 100), cv2.FONT_HERSHEY_SIMPLEX,
                            0.50, (0, 255, 0), 1)
                draw_info()
                cv2.imshow("BIM Data Collection — Desktop view", display)
                key = cv2.waitKey(1)
                if key == 32:
                    target_sequences  += BASE_TARGET_SEQUENCES
                    waiting_for_space  = False
                elif key == 27:
                    break
                continue

            if non_zero_ratio < MIN_NONZERO_RATIO:
                cv2.putText(display, "Low quality — adjust position",
                            (8, 180), cv2.FONT_HERSHEY_SIMPLEX,
                            0.5, (0, 0, 255), 1)
            else:
                if frame_counter % FRAME_SKIP == 0:
                    sequence.append(keypoints)

            if len(sequence) == SEQUENCE_LENGTH:
                seq_array = np.array(sequence, dtype=np.float32)
                sequence  = []
                seq_nz    = np.count_nonzero(seq_array) / (active_features * SEQUENCE_LENGTH)
                if seq_nz < MIN_NONZERO_RATIO:
                    print(f"  ⚠️  Discarding #{counter} (quality {seq_nz:.0%})")
                else:
                    save_sequence(seq_array)
                    if counter >= target_sequences:
                        waiting_for_space = True

            cv2.putText(display, f"Seq: {counter}/{target_sequences}",
                        (8, 100), cv2.FONT_HERSHEY_SIMPLEX,
                        0.55, (0, 255, 0), 1)
            cv2.putText(display,
                        f"Frames: {len(sequence)}/{SEQUENCE_LENGTH}",
                        (8, 116), cv2.FONT_HERSHEY_SIMPLEX,
                        0.45, (0, 255, 255), 1)

            dh = display.shape[0]
            bar_w = int((len(sequence) / SEQUENCE_LENGTH) * 160)
            cv2.rectangle(display, (8, dh - 20), (168, dh - 10),
                          (60, 60, 60), -1)
            cv2.rectangle(display, (8, dh - 20), (8 + bar_w, dh - 10),
                          (0, 200, 255), -1)

            draw_info(extra_y=70)
            cv2.imshow("BIM Data Collection — Desktop view", display)
            if cv2.waitKey(1) == 27:
                break

        # ══════════════════════════════════════════════════════════════════
        # DYNAMIC MODE
        # ══════════════════════════════════════════════════════════════════
        else:
            key = cv2.waitKey(1)

            if state == "IDLE":
                sequence = []
                cv2.putText(display, "SPACE to record | ESC to exit",
                            (8, 100), cv2.FONT_HERSHEY_SIMPLEX,
                            0.50, (0, 255, 255), 1)
                cv2.putText(display, f"Saved: {counter}",
                            (8, 116), cv2.FONT_HERSHEY_SIMPLEX,
                            0.50, (0, 255, 0), 1)
                draw_info()
                cv2.imshow("BIM Data Collection — Desktop view", display)
                if key == 32:
                    state = "COUNTDOWN"
                    countdown_start_time = curr_time
                elif key == 27:
                    break
                continue

            elif state == "COUNTDOWN":
                elapsed   = curr_time - countdown_start_time
                remaining = int(max(0, COUNTDOWN_SECONDS - elapsed)) + 1
                
                cv2.putText(display, f"Ready: {remaining}",
                            (8, 110), cv2.FONT_HERSHEY_SIMPLEX,
                            1.2, (0, 255, 255), 2)
                cv2.putText(display, "Perform sign after countdown",
                            (8, 138), cv2.FONT_HERSHEY_SIMPLEX,
                            0.45, (0, 255, 0), 1)
                draw_info()
                cv2.imshow("BIM Data Collection — Desktop view", display)
                
                if elapsed >= COUNTDOWN_SECONDS:
                    sequence = []
                    state    = "RECORDING"
                    state_start_time = curr_time
                    _last_sample_time = 0.0    
                    
                if key == 27:
                    break
                continue

            elif state == "RECORDING":
                _recording_elapsed = curr_time - (state_start_time or curr_time)

                cv2.putText(display, "● REC",
                            (8, 50), cv2.FONT_HERSHEY_SIMPLEX,
                            0.9, (0, 0, 255), 2)
                cv2.putText(display,
                            f"Frames: {len(sequence)}/{SEQUENCE_LENGTH}",
                            (8, 80), cv2.FONT_HERSHEY_SIMPLEX,
                            0.50, (0, 255, 255), 1)
                cv2.putText(display, f"Saved: {counter}",
                            (8, 96), cv2.FONT_HERSHEY_SIMPLEX,
                            0.45, (0, 255, 0), 1)

                dh    = display.shape[0]
                bar_w = int((len(sequence) / SEQUENCE_LENGTH) * 160)
                cv2.rectangle(display, (8, dh - 20), (168, dh - 10),
                              (60, 60, 60), -1)
                cv2.rectangle(display, (8, dh - 20), (8 + bar_w, dh - 10),
                              (0, 0, 255), -1)

                if non_zero_ratio >= MIN_NONZERO_RATIO:
                    time_since_last = (curr_time - _last_sample_time) * 1000
                    if _last_sample_time == 0 or time_since_last >= SAMPLE_INTERVAL_MS:
                        sequence.append(keypoints)
                        _last_sample_time = curr_time
                else:
                    cv2.putText(display, "Low quality — keep hands visible",
                                (8, 130), cv2.FONT_HERSHEY_SIMPLEX,
                                0.42, (0, 0, 255), 1)

                draw_info(extra_y=70)
                cv2.imshow("BIM Data Collection — Desktop view", display)

                if len(sequence) >= SEQUENCE_LENGTH:
                    seq_array = np.array(
                        sequence[:SEQUENCE_LENGTH], dtype=np.float32)
                    seq_nz = np.count_nonzero(seq_array) / (active_features * SEQUENCE_LENGTH)
                    if seq_nz < MIN_NONZERO_RATIO:
                        print(f"  ⚠️  Discarding dynamic #{counter} "
                              f"(quality {seq_nz:.0%})")
                    else:
                        save_sequence(seq_array)
                    _last_sample_time = 0.0    
                    state = "IDLE"

                if key == 27:
                    break
                continue

cap.release()
cv2.destroyAllWindows()
print(f"\n✅ Done — {counter} sequences saved to {label_path}")

  BIM Sign Language — Desktop Data Collection

💻 Connecting to WebCam Index: 0
✅ Connected

Select mode:
  1 — Static sign  (hold the pose)
  2 — Dynamic sign (moving gesture)

Existing categories:
  1 — Alphabet
  2 — Greet
  0 — Create new category

Configure required regions for this sign:

✅ Setup complete
   Mode             : Static
   Category         : Alphabet
   Label            : Y
   Tracking Face    : ❌ No
   Tracking Pose    : ❌ No

Press SPACE to begin. Press ESC to quit.
  ✅ Saved #0  shape=(8, 204)  nz=50%
  ✅ Saved #1  shape=(8, 204)  nz=50%
  ✅ Saved #2  shape=(8, 204)  nz=50%
  ✅ Saved #3  shape=(8, 204)  nz=50%
  ✅ Saved #4  shape=(8, 204)  nz=50%
  ✅ Saved #5  shape=(8, 204)  nz=50%
  ✅ Saved #6  shape=(8, 204)  nz=50%
  ✅ Saved #7  shape=(8, 204)  nz=50%
  ✅ Saved #8  shape=(8, 204)  nz=50%
  ✅ Saved #9  shape=(8, 204)  nz=50%
  ✅ Saved #10  shape=(8, 204)  nz=50%
  ✅ Saved #11  shape=(8, 204)  nz=50%
  ✅ Saved #12  shape=(8, 204)  nz=50%
  ✅ Saved #13  shape=(8,

In [ ]:
import cv2

PHONE_CAMERA_URL = 'http://172.20.10.5:8080/video'
cap = cv2.VideoCapture(PHONE_CAMERA_URL)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Apply your confirmed rotation first
    rotated = cv2.rotate(frame, cv2.ROTATE_90_COUNTERCLOCKWISE)

    # All four flip combinations
    no_flip   = rotated.copy()
    h_flip    = cv2.flip(rotated, 1)   # horizontal (left-right)
    v_flip    = cv2.flip(rotated, 0)   # vertical   (up-down)
    both_flip = cv2.flip(rotated, -1)  # both

    h = 280
    def resize(img):
        ratio = h / img.shape[0]
        return cv2.resize(img, (int(img.shape[1] * ratio), h))

    r_nf = resize(no_flip)
    r_hf = resize(h_flip)
    r_vf = resize(v_flip)
    r_bf = resize(both_flip)

    row = cv2.hconcat([r_nf, r_hf, r_vf, r_bf])

    # Labels
    w = r_nf.shape[1]
    cv2.putText(row, "No flip",    (10,          25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
    cv2.putText(row, "H flip",     (w + 10,      25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
    cv2.putText(row, "V flip",     (w*2 + 10,    25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
    cv2.putText(row, "Both flips", (w*3 + 10,    25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    cv2.imshow("Raise RIGHT hand — it should appear on RIGHT side (ESC=quit)", row)
    if cv2.waitKey(1) == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2

PHONE_CAMERA_URL = 'http://172.20.10.5:8080/video'
cap = cv2.VideoCapture(PHONE_CAMERA_URL)
ret, frame = cap.read()

print(f"Raw from IP Webcam       : {frame.shape[1]}×{frame.shape[0]}px")

rotated = cv2.rotate(frame, cv2.ROTATE_90_COUNTERCLOCKWISE)
print(f"After 270° rotation      : {rotated.shape[1]}×{rotated.shape[0]}px")
print(f"Flutter sensor (effW×effH): 480×720px")
print(f"Match: {rotated.shape[1] == 480 and rotated.shape[0] == 720}")

cap.release()